# 05 · Linear Regression
**Owner:** Senanayake · **Branch:** `model/linear-regression`

GroupKFold CV on the training set (split by battery) + fit on full train + predict on the held-out test set. Metrics saved for notebook 10.


In [ ]:
# --- Colab setup (run first) ---
import os
if not os.path.exists('ev-battery-rul-prediction'):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git
%cd ev-battery-rul-prediction
!git pull -q
!pip install -q -r requirements.txt
import sys; sys.path.append('.')
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
import os, json, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
train = pd.read_parquet(os.path.join(DATA_DIR,'train.parquet'))
test  = pd.read_parquet(os.path.join(DATA_DIR,'test.parquet'))
feats = json.load(open('outputs/feature_list.json'))
Xtr, ytr, gtr = train[feats], train['RUL'], train['cell']
Xte, yte = test[feats], test['RUL']


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression
model = make_pipeline(RobustScaler(), LinearRegression())


In [ ]:
# cross-validation on train (grouped by battery)
gkf = GroupKFold(n_splits=5); maes, r2s = [], []
for tr, va in gkf.split(Xtr, ytr, gtr):
    model.fit(Xtr.iloc[tr], ytr.iloc[tr])
    p = model.predict(Xtr.iloc[va])
    maes.append(mean_absolute_error(ytr.iloc[va], p)); r2s.append(r2_score(ytr.iloc[va], p))
print(f'CV  MAE {np.mean(maes):.1f}+/-{np.std(maes):.1f}  R2 {np.mean(r2s):.3f}')


In [ ]:
# fit on full train, evaluate on held-out test
model.fit(Xtr, ytr); pte = model.predict(Xte)
test_mae = mean_absolute_error(yte, pte); test_r2 = r2_score(yte, pte)
print(f'TEST MAE {test_mae:.1f}  R2 {test_r2:.3f}')


In [ ]:
os.makedirs('outputs', exist_ok=True)
metrics = {'model':'Linear Regression','member':'Senanayake',
   'cv_MAE':float(np.mean(maes)),'cv_R2':float(np.mean(r2s)),
   'test_MAE':float(test_mae),'test_R2':float(test_r2)}}
json.dump(metrics, open('outputs/metrics_05.json','w'), indent=2)
pd.DataFrame({'cell':test['cell'],'RUL':yte,'pred':pte}).to_csv('outputs/pred_05.csv', index=False)
metrics
